In [152]:

from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph import graph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
import time
from pprint import pprint
load_dotenv()

True

In [153]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

In [154]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [155]:
def generate_joke(state: JokeState) -> JokeState:
    response = llm.invoke(f"Generate a funny and original joke on the topic: {state["topic"]}. Reply with only the joke.")
    return {"joke": response.content}

In [156]:
def explain_joke(state: JokeState) -> JokeState:
    response = llm.invoke(f"Explain the joke: {state["joke"]}. Reply with only joke explaination")
    return {"explanation": response.content}

In [157]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

In [158]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [159]:
initial_state = {"topic": "AI"}
thread_id = "thread_1"
config = {"configurable": {"thread_id": thread_id}}
final_state = workflow.invoke(input = initial_state, config = config)

final_state["joke"]


'I asked my AI to write a love letter, but it just copied and pasted. Turns out, it had a "Ctrl" over its emotions and couldn’t compute new feelings.'

In [160]:
# list(workflow.get_state_history(config))

print("=== State History ===")
for snapshot in workflow.get_state_history(config):
    print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
    print("Next:  ", snapshot.next)
    print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
                     for k, v in snapshot.values.items()})
    print()


=== State History ===
--- Step 2 | 2026-06-17T11:14:40.869473+00:00 ---
Next:   ()
Values: {'topic': 'AI', 'joke': 'I asked my AI to write a love letter, but it just copied and...', 'explanation': 'The joke uses wordplay on "Ctrl" (short for "Control," as in...'}

--- Step 1 | 2026-06-17T11:14:39.009089+00:00 ---
Next:   ('explain_joke',)
Values: {'topic': 'AI', 'joke': 'I asked my AI to write a love letter, but it just copied and...'}

--- Step 0 | 2026-06-17T11:14:35.037276+00:00 ---
Next:   ('generate_joke',)
Values: {'topic': 'AI'}

--- Step -1 | 2026-06-17T11:14:35.026389+00:00 ---
Next:   ('__start__',)
Values: {}



In [161]:
workflow.invoke(None, config = config)

{'topic': 'AI',
 'joke': 'I asked my AI to write a love letter, but it just copied and pasted. Turns out, it had a "Ctrl" over its emotions and couldn’t compute new feelings.',
 'explanation': 'The joke uses wordplay on "Ctrl" (short for "Control," as in keyboard shortcuts like Ctrl+C/V for copy/paste) to mock an AI\'s lack of originality. The AI "couldn\'t compute new feelings" because it can only replicate existing data (copy-paste), not generate genuine emotions. The pun ties "Ctrl" to both the keyboard key and the idea of "controlling" emotions, highlighting the AI\'s mechanical, unfeeling nature.'}